# Py12 - K-Means Neighbors Clustering

You are going to cluster states through K-Means clustering based on their neighbors. The dataset contains information about each state.

### Step 1: Load and Explore the Data

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix


# Load the statelife.csv file. Hint: how many rows to skip?
df = pd.read_csv(?)

# drop rows with missing values
df = df.dropna()

df

### Step 2: Standardize the Features

Standardization is crucial for K-Means because the algorithm is distance-based. Features with larger scales will dominate the distance calculations.

In [ ]:
# Standardize the features
X = df.select_dtypes(include=[np.number]).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original data - first sample:")
print(X[0])
print("\nStandardized data - first sample:")
print(X_scaled[0])
print("\nMean of standardized features (should be ~0):")
print(X_scaled.mean(axis=0))
print("\nStd of standardized features (should be ~1):")
print(X_scaled.std(axis=0))

### Step 3: Find Optimal Number of Clusters (Elbow Method)

The Elbow Method helps determine the optimal number of clusters (K) by plotting the inertia (within-cluster sum of squares) against different K values.

The inertia is the *sum of squared distances* between each *point* and its assigned *cluster center*. As K increases, inertia decreases because points are closer to their cluster centers. The goal is to find the "elbow" point where the rate of decrease sharply changes, indicating a suitable K.


In [ ]:
# Plot inertia for different numbers of clusters
import seaborn as sns
import matplotlib.pyplot as plt

inertias = []
range_tested = range(1, 10)

for k in range_tested:
    # K-Means clustering function uses a given number of clusters, a random variable for reproducibility, 
    # and n_init for stability (number of initializations, meaning the algorithm will run multiple times 
    # with different centroid seeds)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot the elbow curve as a single plot
sns.lineplot(x=range_tested, y=inertias, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.show()

### Step 4: Build and Fit the K-Means Model

In [ ]:
# Create and fit K-Means model 
# You will need to pick the number of clusters based on the elbow plot (look for the "elbow" point where inertia starts to decrease more slowly)
kmeans = KMeans(n_clusters=?, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to the dataframe
df['cluster'] = cluster_labels

print("Cluster Centers (standardized):")
print(kmeans.cluster_centers_)
print("\nCluster Distribution:")
print(df['cluster'].value_counts().sort_index())
print("\nSilhouette Score:", silhouette_score(X_scaled, cluster_labels))

### Step 5: Visualize the Clusters

In [ ]:
# Create a map of the US with states colored by their assigned cluster
# %pip install geopandas shapely
import geopandas as gpd
from shapely.geometry import Point

# Load US states shapefile using the updated method
us_states = gpd.read_file('https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip')
us_states = us_states[us_states['iso_3166_2'].str.startswith('US-')]

# Filter for contiguous US states only (exclude Alaska, Hawaii, and territories)
excluded_states = ['Alaska', 'Hawaii']
us_states = us_states[~us_states['name'].isin(excluded_states)]

# Merge cluster data with geospatial data
us_states = us_states.merge(df, left_on='name', right_on='statename', how='left')

# Plot the states colored by cluster with borders
fig, ax = plt.subplots(1, 1, figsize=(20, 12))
us_states.plot(column='cluster', ax=ax, legend=True, cmap='Set3', 
               edgecolor='black', linewidth=0.5,
               missing_kwds={"color": "lightgrey", "edgecolor": "black", "linewidth": 0.5})
plt.title('US States Clustered by Life Expectancy Data', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()